# 06장 보안 실습 — 프로세스·소켓 문맥 연결


## Goal

합성 스냅샷을 연결하고 PID·시각·서비스 문맥의 한계를 설명합니다.

[교안과 분석 질문](../../06-system-inspection/06-3-host-process-investigation.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-06-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'processes.psv': 'pid|ppid|user|started_kst|exe|unit\n1|0|root|2026-09-10T08:00:00+09:00|/usr/lib/systemd/systemd|init.scope\n200|1|root|2026-09-10T08:01:00+09:00|/usr/sbin/sshd|ssh.service\n410|200|analyst|2026-09-10T09:02:00+09:00|/usr/bin/bash|session-4.scope\n520|1|collector|2026-09-10T09:05:00+09:00|/opt/collector/bin/report|report-helper.service\n', 'sockets.psv': 'state|local|peer|pid\nLISTEN|0.0.0.0:22|0.0.0.0:*|200\nESTAB|192.0.2.20:22|192.0.2.10:50103|200\nESTAB|192.0.2.20:41000|203.0.113.7:443|520\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 프로세스 4행, PID 520의 서비스·목적지 연결, 판정은 추가 검토

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 필드와 프로세스 수 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
head -n 1 "$COURSE_DATA/processes.psv"
awk -F '|' 'NR>1 {print $1, $2, $3, $5}' "$COURSE_DATA/processes.psv" > "$COURSE_OUT/process-preview.txt"
test "$(wc -l < "$COURSE_OUT/process-preview.txt")" -eq 4
printf 'process_rows=4\n'


### 2. PID로 연결하되 시각 한계를 유지


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR==FNR {if (FNR>1) exe[$1]=$5; next}
 FNR>1 {print $1 "|" $3 "|" $4 "|" (($4 in exe) ? exe[$4] : "unknown")}' \
 "$COURSE_DATA/processes.psv" "$COURSE_DATA/sockets.psv" > "$COURSE_OUT/linked.psv"
grep -Fx 'ESTAB|203.0.113.7:443|520|/opt/collector/bin/report' "$COURSE_OUT/linked.psv"


### 3. 부모와 서비스 문맥 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $1==520 {print "parent=" $2 " user=" $3 " unit=" $6}' \
 "$COURSE_DATA/processes.psv" > "$COURSE_OUT/context.txt"
grep -Fx 'parent=1 user=collector unit=report-helper.service' "$COURSE_OUT/context.txt"
printf 'verdict=needs_service_and_destination_review\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: 익숙한 서비스 이름이 신뢰의 근거인가

**Red Team 질문:** 프로세스·서비스 정보에서 권한 경계나 운영상의 노출을 이해할 수 있는가? 실행 이름이 아니라 사용자·실행 파일·설정 출처가 중요하며, 버전 정보만으로 취약점 악용 가능성을 확정하지 않습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 환경과 실행 권한을 이해하려는 목적. 관찰 범위·네임스페이스·읽기 권한이 전제 |
| Command / Observation | ps·pgrep·/proc·systemctl 정보를 비교. 합성 PID 520은 collector로 09:05 시작 |
| System Change / Artifact | 프로세스 생성·부모 관계·실행 파일·FD. 종료된 프로세스는 현재 /proc에 없을 수 있음 |
| Log prerequisite | 사전 exec 감사/EDR와 서비스 Journal. 프로세스 스냅샷은 과거 실행 전부가 아님 |
| Blue Team Investigation | report-helper.service의 unit·drop-in·실행 경로·배포 이력을 PID와 비교 |
| Detection | 예상 사용자·실행 경로·부모·시간대에서 벗어난 조합을 검토. 이름만 차단하지 않음 |
| Mitigation | 서비스별 최소 권한·신뢰된 배포 경로·필요한 실행 기록. 중지 전 휘발성 근거와 영향 검토 |

**반례와 해설:** PPID 1은 정상 서비스에도 나타납니다. PID 520이라는 숫자는 다음 부팅에서 다른 프로세스가 재사용할 수 있습니다. 지금 실행 중인 파일의 해시와 과거 사건 시점 파일의 해시를 같은 사실로 취급하지 않습니다.

**제출 과제:** PID 520에 대해 사용자·시작 시각·실행 경로·서비스 네 필드를 제시합니다. Red Team은 추가 확인할 경계 하나, Blue Team은 필요한 자료 두 개를 적습니다. 프로세스 명령행 전체를 공개 보고서에 복사하기 전에 민감정보 검토 필요성도 설명합니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: 수신 포트와 외부 연결의 의미

**Red Team 질문:** 로컬 주소·라우트·수신 상태가 어느 접근 경계를 보여주는가? 내부 이동이나 피벗 가능성에는 실제 접근 통제 등 추가 조건이 필요합니다. 이 실습은 로컬 자료의 의미를 검토하며 원격 탐색·터널 구성은 하지 않습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 서비스 접근·통신 경계 이해. 라우팅·방화벽·NAT·서비스 인증은 별도 조건 |
| Command / Observation | ip·ss·lsof와 프로세스 사본을 연결. PID 520의 peer는 203.0.113.7:443 |
| System Change / Artifact | 소켓 생성·연결 상태. 소켓 한 행은 전송 내용·명령 실행을 담지 않음 |
| Log prerequisite | DNS·프록시·방화벽 flow·EDR 등의 실제 수집 여부. ss를 실행했다고 과거 통신 로그가 생기지 않음 |
| Blue Team Investigation | 프로세스 신원·승인 목적지·통신 기간·전송량·관련 서비스 변경 검토 |
| Detection | 예상 밖 프로세스와 목적지의 조합, 시간 패턴, 여러 기록의 일치로 조사 후보 구성 |
| Mitigation | 승인된 통신 범위·서비스 노출·원격 관리 정책 검토. 차단은 증거 보존과 업무 영향 고려 |

**반례와 해설:** 정상 모니터링 프로그램도 주기적으로 443에 연결합니다. 반대로 443이라고 내용이 TLS이거나 안전하다고 보장되지 않습니다. C2·Reverse Shell은 조사 가설이지 이 사본의 결론이 아닙니다. 목적지 IP를 확인하려고 수업에서 실제로 접속하지 않습니다.

**제출 과제:** LISTEN 한 행과 ESTAB 한 행의 차이를 설명하고 각각 가능한 정상 업무를 적습니다. 경보에 사용할 필드 세 개와 현재 없는 필드 두 개를 구분합니다. 서로 다른 호스트의 같은 PID를 조인하지 않는 기준을 적으면 통과입니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
